# RetailMart Sales Analysis
This notebook performs the full data pipeline: Load,Clean, Transform ,Load to SQL Report

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

BASE = r'D:\Retail Mart Sales Analysis'# here we have enter the location and name of the folder in read mode which is assigned as value you can change it with your folder location

## Task 1: Data Ingestion

In [ ]:
# 1. Load all three CSV files into pandas DataFrames
sales = pd.read_csv(f'{BASE}\\sales_data.csv')
products = pd.read_csv(f'{BASE}\\product.csv')
stores = pd.read_csv(f'{BASE}\\stores.csv')

print('--- sales_data.csv ---')
print(f'Shape: {sales.shape}')
print(sales.head())

print('\n--- products.csv ---')
print(f'Shape: {products.shape}')
print(products.head())

print('\n--- stores.csv ---')
print(f'Shape: {stores.shape}')
print(stores.head())

In [ ]:
# 2. Check for missing values
print('--- sales_data missing values ---')
print(sales.isnull().sum())
print('\n--- products missing values ---')
print(products.isnull().sum())
print('\n--- stores missing values ---')
print(stores.isnull().sum())

## Task 2: Data Cleaning

In [ ]:
# 3. Remove duplicate rows
dup_count = sales.duplicated().sum()
print(f'Duplicates found: {dup_count}')
sales_clean = sales.drop_duplicates().copy()
print(f'Duplicates removed: {dup_count}')

In [ ]:
# 4. Fill missing quantity with 0, drop rows where amount is NULL
sales_clean['quantity'] = pd.to_numeric(sales_clean['quantity'], errors='coerce').fillna(0).astype(int)
sales_clean['amount'] = pd.to_numeric(sales_clean['amount'], errors='coerce')
sales_clean = sales_clean.dropna(subset=['amount'])
print('Cleaned DataFrame:')
print(sales_clean)
print(f'\nCleaned DataFrame shape: {sales_clean.shape}')

In [ ]:
# 5. Convert sale_date to datetime, amount to float
sales_clean['sale_date'] = pd.to_datetime(sales_clean['sale_date'], format='mixed', dayfirst=True)
sales_clean['amount'] = sales_clean['amount'].astype(float)

print('Data types after conversion:')
print(sales_clean.dtypes)
print()
print(sales_clean.head())

## Task 3: Data Transformation

In [ ]:
# 6. Merge all three DataFrames
sales_clean['product_id'] = sales_clean['product_id'].astype(str)
products['product_id'] = products['product_id'].astype(str)

merged = sales_clean.merge(products, on='product_id', how='left').merge(stores, on='store_id', how='left')
print('Final Merged DataFrame:')
merged

In [ ]:
# 7. Add total_revenue = quantity * price
merged['total_revenue'] = merged['quantity'] * merged['price']
rev = merged['total_revenue'].dropna()
print(f'Mean: {np.mean(rev):.2f}')
print(f'Max:  {np.max(rev):.2f}')
print(f'Min:  {np.min(rev):.2f}')

In [ ]:
# 8. Group by city, total revenue, sort descending
city_revenue = merged.groupby('city')['total_revenue'].sum().sort_values(ascending=False)
print('Total Revenue per City:')
city_revenue

## Task 4: Data Loading (SQL)

In [ ]:
# 9. Load merged DataFrame into SQLite
conn = sqlite3.connect(f'{BASE}\\retail_mart.db')
merged.to_sql('retail_sales', conn, if_exists='replace', index=False)
print('Data loaded into retail_sales table')

In [ ]:
# 10. SQL query: Top 3 best-selling products by quantity
query_top3 = '''
SELECT product_name, SUM(quantity) as total_qty_sold
FROM retail_sales
GROUP BY product_name
ORDER BY total_qty_sold DESC
LIMIT 3
'''
pd.read_sql(query_top3, conn)

## Task 5: Reporting & Insights

In [ ]:
# 11. SQL query: Total revenue per store per day
query_rev = '''
SELECT store_id, store_name, sale_date, SUM(total_revenue) as revenue
FROM retail_sales
GROUP BY store_id, store_name, sale_date
ORDER BY store_id, sale_date
'''
pd.read_sql(query_rev, conn)

In [ ]:
# 12. Summary report
print(f'Total number of transactions: {len(merged)}')
print(f'Total revenue: {merged["total_revenue"].sum():.2f}')
print(f'Top selling city: {city_revenue.index[0]}')
top_prod = merged.groupby('product_name')['total_revenue'].sum().idxmax()
print(f'Top selling product: {top_prod}')
conn.close()

## Task 6: Pipeline & Error Handling

In [ ]:
# 13. run_pipeline() with error handling
def run_pipeline(sales_path, products_path, stores_path, db_path):
    try:
        sales = pd.read_csv(sales_path)
        products = pd.read_csv(products_path)
        stores = pd.read_csv(stores_path)

        sales_clean = sales.drop_duplicates().copy()
        sales_clean['quantity'] = pd.to_numeric(sales_clean['quantity'], errors='coerce').fillna(0).astype(int)
        sales_clean['amount'] = pd.to_numeric(sales_clean['amount'], errors='coerce')
        sales_clean = sales_clean.dropna(subset=['amount'])
        sales_clean['sale_date'] = pd.to_datetime(sales_clean['sale_date'], format='mixed', dayfirst=True)
        sales_clean['amount'] = sales_clean['amount'].astype(float)
        sales_clean['product_id'] = sales_clean['product_id'].astype(str)

        products['product_id'] = products['product_id'].astype(str)
        merged = sales_clean.merge(products, on='product_id', how='left').merge(stores, on='store_id', how='left')
        merged['total_revenue'] = merged['quantity'] * merged['price']

        conn = sqlite3.connect(db_path)
        merged.to_sql('retail_sales', conn, if_exists='replace', index=False)
        conn.close()

        print(f'Pipeline completed. {len(merged)} records loaded.')
        return merged

    except FileNotFoundError as e:
        print(f'ERROR: File not found - {e}')
    except Exception as e:
        print(f'ERROR: {e}')

In [ ]:
# 14. Run the pipeline
result = run_pipeline(
    f'{BASE}\\sales_data.csv',
    f'{BASE}\\product.csv',
    f'{BASE}\\stores.csv',
    f'{BASE}\\retail_mart.db'
)
print('--- Pipeline Execution Complete ---')